##### collapse

In [37]:
import torch
from facenet_pytorch import MTCNN, InceptionResnetV1
import os
from PIL import Image
import cv2
import torch.nn.functional as F

In [38]:
device = "cuda" if torch.cuda.is_available() else "cpu"

mtcnn = MTCNN(image_size=160, margin=0, device=device, keep_all=True)

resnet = InceptionResnetV1(pretrained='vggface2').eval().to(device)

We now improve identity representation.

Your current database:

`person → embedding (from 1 image)`

Problem: <br>
One image is unstable.

Lighting, angle, expression → embedding shifts.

### Identity Is A Distribution

Think of a person’s face not as:

`one point in 512D space`

But as:

`a cluster of points`

Each image produces slightly different embedding.

So we want:

`identity_center = average of all embeddings`

This reduces noise.

In [ ]:
def build_database(folder_path):
    
    database = {}
    
    for person in os.listdir(folder_path):
        
        person_path = os.path.join(folder_path, person)
        
        if not os.path.isdir(person_path):
            continue
        
        embeddings = []
        
        for file in os.listdir(person_path):
            
            img_path = os.path.join(person_path, file)
            
            img = Image.open(img_path)
            
            face = mtcnn(img)
            
            if face is None:
                continue
            
            if face.ndim == 3:
                face = face.unsqueeze(0)
                
            face = face.to(device)
            
            with torch.no_grad():
                embedding = resnet(face)
                
            embedding /= embedding.norm(dim=1, keepdim=True)
            
            embeddings.append(embedding)
            
        if len(embeddings) > 0:
            stacked = torch.cat(embeddings, dim=0)
            
            mean_embedding = torch.mean(stacked, dim=0, keepdim=True)
            
            mean_embedding /= mean_embedding.norm(dim=1, keepdim=True)
            
            database[person] = mean_embedding
            
    return database

#### `torch.cat(embeddings, dim=0)`

If you have:

    [1x512]
    [1x512]
    [1x512]

`cat(dim=0)` gives:

`3x512`

It stacks embeddings vertically.

`torch.mean(stacked, dim=0)`

- Computes mean across rows.

If stacked is:

`3 x 512`

Mean across dim=0 gives:

`512`

Then `keepdim=True` keeps it:

`1 x 512`

Which matches model output shape.

#### Why Normalize Again After Mean?

Averaging changes vector length.

If you don’t normalize: <br>
Cosine similarity becomes inconsistent.

So we re-normalize.

### Recognition Function and Webcam Loop (same as before)

In [40]:
def recognize_faces(frame, database, threshold=0.6):

    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    img = Image.fromarray(frame_rgb)

    boxes, _ = mtcnn.detect(img)

    if boxes is None:
        return frame

    faces = mtcnn(img)

    if faces is None:
        return frame

    if faces.ndim == 3:
        faces = faces.unsqueeze(0)

    faces = faces.to(device)

    with torch.no_grad():
        embeddings = resnet(faces)

    # Normalize embeddings
    embeddings = embeddings / embeddings.norm(dim=1, keepdim=True)

    for i, embedding in enumerate(embeddings):

        embedding = embedding.unsqueeze(0)

        best_match = None
        max_similarity = -1

        for name, db_embedding in database.items():

            similarity = F.cosine_similarity(
                embedding,
                db_embedding
            ).item()

            if similarity > max_similarity:
                max_similarity = similarity
                best_match = name

        if max_similarity > threshold:
            label = f"{best_match} ({max_similarity:.2f})"
            color = (0, 255, 0)
        else:
            label = "Unknown"
            color = (0, 0, 255)

        x1, y1, x2, y2 = map(int, boxes[i])
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
        cv2.putText(
            frame,
            label,
            (x1, y1 - 10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            color,
            2
        )

    return frame

In [41]:
database = build_database("faces/single")

cap = cv2.VideoCapture(0)

while True:

    ret, frame = cap.read()

    if not ret:
        break

    frame = recognize_faces(frame, database)

    cv2.imshow("Face Recognition", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()